In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from src.imputation import imputation_normal_distribution, log2
from src.correlation import pairwise_correlation
from src.data_processing import pre_processing, calculate_data_completeness, filter_data, summarize_filtered_data
from src.data_processing import compute_pca,generate_pca_plot, calculate_cv
from src.data_processing import generate_pca_plot1
from src.multi_variate_prediction import get_features, get_prediction_score
from src.statistical_testing import perform_linear_regression, pg_ttest
import pingouin as pg
from matplotlib_venn import venn3, venn3_circles
from venn import venn
import statsmodels.stats.multitest as multi
from tqdm import tqdm
from scipy.stats import pearsonr
from scipy.stats import zscore
import pickle
import string
from adjustText import adjust_text

In [ ]:
from mycolorpy import colorlist as mcp
colors = mcp.gen_color(cmap="Paired", n=10)

### Proteomics data processing

#### Define directories

In [ ]:
import os
from pathlib import Path

# Get the number of available CPUs
CPUS = os.cpu_count()

# Define the paths for the raw and processed data folders
DATA_FOLDER_2k = '/Volumes/auditing-groupdirs/SUN-CPR-TARGET_PROTEOMICS/2k_discovery'
DATA_FOLDER_RAW = Path(os.path.join(DATA_FOLDER_2k, 'data/raw'))
DATA_FOLDER_PROCESSED = Path(os.path.join(DATA_FOLDER_2k, 'data/processed'))
DATA_FOLDER_CLINIC = '/Volumes/auditing-groupdirs/SUN-CBMR-Childhood-Genetic-TCOC/Proteomics analysis/GitHub/TARGET/'

# Ensure base folders are created and define subfolder paths
os.makedirs(DATA_FOLDER_PROCESSED, exist_ok=True)
subfolders = ['tables', 'results', 'figures', 'pQTL', 'dash', 'annotations', 'GWAS']
folders = {f: Path(DATA_FOLDER_2k, f) for f in subfolders}

#Create subfolders if they don't exist
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)
    
Path(folders['pQTL'] / 'gemma').mkdir(exist_ok=True)
gemma_path = Path(folders['pQTL'], 'gemma')

#### Import data and format column headers

In [ ]:
# Read annotation file
annotation_file = pd.read_csv(os.path.join(DATA_FOLDER_RAW, 'annotations.csv'), sep=';')

# Read Spectronaut output report into a DataFrame
# 2K@24 LIB4P OneGo
report_filename= '20240229_010223_Protein Lili long (Normal)_canonical_specific.tsv'

cols_to_keep = ['R.FileName', 'PG.Genes', 'PG.ProteinAccessions', 'PG.Quantity']

RE_READ = False
if not RE_READ:
    data_raw_long = pd.read_pickle(os.path.join(DATA_FOLDER_PROCESSED, 'data_raw_long.pkl'))
    
else:   
    chunk_size = 10000
    file_to_read = pd.read_csv(os.path.join(DATA_FOLDER_RAW, report_filename), 
                               delimiter='\t', na_values='Filtered', usecols=cols_to_keep, chunksize=chunk_size)
    chunks = []
    for chunk in tqdm(file_to_read):
        chunks.append(chunk)
    report_plasma = pd.concat(chunks)
    data_raw_long = report_plasma.dropna().reset_index().drop(['index'], axis=1)
    with open(os.path.join(DATA_FOLDER_PROCESSED, 'data_raw_long.pkl'), 'wb') as pfile:
        pickle.dump(data_raw_long, pfile, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
data_raw_long = pre_processing(data_raw_long)
protein_ids = data_raw_long[['Gene names', 'Protein IDs', 
                             'Gene name', 'Protein ID',
                             'ProteinID_Genename']].drop_duplicates()

In [ ]:
cond1 = data_raw_long['Sample ID']=='Plate1_1'
cond2 = data_raw_long['Protein ID']=='A0A075B6H7'
test_value=data_raw_long[(cond1) & (cond2)]['PG.Quantity'].iloc[0]
assert abs(test_value-1829.426757)<1e-4, 'Value changed compared to previous run'

In [ ]:
#Prepare ID list for genome coordinate mapping
protein_ids['Protein IDs_list'] = protein_ids['Protein IDs'].str.split(';')
protein_ids_all = protein_ids.explode('Protein IDs_list')

with open(gemma_path / 'protein_id_all.list', 'w') as file:
    for protein in protein_ids_all['Protein IDs_list']:
        file.write(str(protein) + '\n')
        
protein_ids_all.to_csv(gemma_path / 'protein_id_all.txt', index=False, sep='\t')

In [ ]:
cols_to_keep = ['Sample ID', 'ProteinID_Genename', 'PG.Quantity']
data_plasma_raw = data_raw_long[cols_to_keep].pivot(columns='Sample ID', index='ProteinID_Genename', values='PG.Quantity')

In [ ]:
df_raw_comp = calculate_data_completeness(data_plasma_raw)
sns.lineplot(data=df_raw_comp, x='rank', y='%Complete')

#### Filter data based on data completeness

In [ ]:
proteins, sample_ids, data_plasma_filtered = filter_data(data_plasma_raw)

#### Summarize filtered data

In [ ]:
summarize_filtered_data(data_plasma_filtered)

#### Plot proteins by data completeness

In [ ]:
df_filtered_comp = calculate_data_completeness(data_plasma_filtered)
sns.lineplot(x='rank', y='%Complete', data=df_filtered_comp)
plt.ylim(0, 1.1)

In [ ]:
data_plasma_filtered_log = data_plasma_filtered.apply(log2)

### Double key between genomics and proteomics data

In [ ]:
# Read the double ID file into a DataFrame and set the index
double_id_file = pd.read_csv(os.path.join(DATA_FOLDER_RAW, 'Target_sampleID.csv'), sep=';', index_col='blood_sample_ID').dropna(how='all')

# Drop rows with all missing values
double_id_file = double_id_file.dropna(how='all')

# Create a dictionary to map blood sample IDs to proteomics sample IDs and vice versa
IDmapping_bloodSampleID_to_SampleID = double_id_file['Sample ID'].to_dict()
IDmapping_SampleID_to_bloodSampleID = dict(zip(double_id_file['Sample ID'], double_id_file.index))

In [ ]:
# Define the path to the output file
Path(os.path.join(folders['pQTL'], 'phenomics')).mkdir(exist_ok=True)
output_file_path = os.path.join(folders['pQTL'], 'phenomics', 'IDmapping_SampleID_to_bloodSampleID.p')

# Dump the dictionary to the output file
with open(output_file_path, 'wb') as output_file:
    pickle.dump(IDmapping_SampleID_to_bloodSampleID, output_file)

### Clinical data wrangling

#### Import data and add relevant columns

In [ ]:
# Read in the raw clinical data file
data_cli_raw = pd.read_csv(os.path.join(DATA_FOLDER_CLINIC, 'HOL_dataset_massspec_bas_fu_clean_v2022.07.07.csv'), 
                          sep=';', decimal=',', low_memory=False)

# Drop unnecessary columns and rows with all missing values
data_cli_raw.drop(['Unnamed: 0'], axis=1, inplace=True)
data_cli_raw.dropna(how='all', axis=1, inplace=True)

# Rename columns and add a proteomics analysis date column
data_cli_raw.rename({'gender':'sex'}, axis=1, inplace=True)
data_cli_raw['proteomics_analysis_date'] = pd.Timestamp('2021-07-17')

# Assign groups of overweight and normalweight 
data_cli_raw['obesity'] = np.where(data_cli_raw['z_BMI.Nysom']>=1.28, 1, 0)

# Add a column for the time between blood sample collection and analysis
#data_cli_raw['blood_sample_date_ts']=[pd.Timestamp(s) for s in data_cli_raw['blood_sample_date']]
data_cli_raw['blood_sample_date_ts'] = pd.to_datetime(data_cli_raw['blood_sample_date'])
storagetime = data_cli_raw['proteomics_analysis_date'] - data_cli_raw['blood_sample_date_ts']
data_cli_raw['time_to_analysis'] = storagetime.dt.days

# Drop any duplicated rows and save the filtered DataFrame to a variable
data_cli_filtered = data_cli_raw[~data_cli_raw.index.duplicated(keep=False)]

print('Filtered clinical data shape: {}'.format(data_cli_filtered.shape))

#### Extract clinical data for samples with proteomics measurement

In [ ]:
data_cli_prot = pd.DataFrame(data= data_plasma_raw.T.index, columns=['Sample ID']).set_index('Sample ID')
data_cli_prot['blood_sample_ID'] = data_cli_prot.index.map(IDmapping_SampleID_to_bloodSampleID)
data_cli_prot = data_cli_prot.reset_index().merge(data_cli_filtered, how='left', left_on='blood_sample_ID', 
                                                  right_on='biobank_ID').set_index('Sample ID')
data_cli_prot.drop('blood_sample_ID', axis=1, inplace=True)

data_cli_prot['age_int']=data_cli_prot['age'].round(0)
bin_numbers_bmi = pd.qcut(
    x=data_cli_prot['BMI'], q=20, labels=False, duplicates='drop'
)
data_cli_prot['bin_numbers_bmi'] = bin_numbers_bmi

In [ ]:
dates = [i for i in data_cli_prot['visit_date'] if type(i)==str]
dates_year = [int(i.split('-')[0]) for i in dates]
data_cli_prot['visit_date_year'] = data_cli_prot['visit_date'].map(dict(zip(dates, dates_year)))
data_cli_prot['visit_date_year_binary'] = np.where(data_cli_prot['visit_date_year']>2015, '>2015', '≤2015')

df = data_cli_prot.copy()

prot_batch = pd.get_dummies(annotation_file.set_index('Sample ID')['Grouping_batch'])
prot_batches = prot_batch.columns.tolist()
df = df.join(prot_batch)
data_cli_prot = df.copy()

In [ ]:
# Save clinical data for phenomics analysis 
data_cli_prot.reset_index().to_csv(folders['pQTL'] / 'phenomics/data_cli_prot.csv', index=False)

### Baseline participant characteristics

In [ ]:
data_cli_base = data_cli_prot.copy()
data_cli_base = data_cli_base[data_cli_base['2k_QA']==0]

para_toinclude = ['obesity', 'age_year', 'sex', 'tanner_stage', 'pubertal_status2', 'BMI','z_BMI.Nysom', 
                  'ALAT', 'ASAT', 'GGT', 'glucose', 'insulin', 'HbA1c', 'triglycerides', 'chol_total',
                  'chol_ldl', 'chol_hdl',]

data_cli_base = data_cli_base[para_toinclude]

In [ ]:
median = data_cli_base.groupby('obesity').median().T
median.columns = [str(i) + '_median' for i in median.columns]
q1 = data_cli_base.groupby('obesity').quantile(0.25).T
q1.columns = [str(i)+ '_q1' for i in q1.columns]
q3 = data_cli_base.groupby('obesity').quantile(0.75).T
q3.columns = [str(i)+ '_q3' for i in q3.columns]
df_median = pd.concat([median, q1, q3], axis=1)[['0.0_median', '0.0_q1', '0.0_q3', '1.0_median', '1.0_q1', '1.0_q3']]

In [ ]:
pd.options.display.float_format = "{:,.1f}".format
data_cli_base.groupby(['obesity', 'sex']).std().T.round(1)

#### Compute PCA

In [ ]:
X_train = data_plasma_filtered_log.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]

In [ ]:
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
df_pc = df_pc.join(data_cli_prot[['time_to_analysis','obesity']])

#### PCA plot

In [ ]:
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2)
fig_pca.savefig(os.path.join(folders['figures'], 'PCA_step1.png'), bbox_inches='tight', dpi=120)
plt.rcParams['pdf.fonttype'] = 42  
fig_pca.savefig('figures/PCA_step1.pdf', bbox_inches='tight', dpi=120)

In [ ]:
qc_markers_platelet = pd.read_excel(os.path.join(folders['annotations'], 'plasma_quality_markers.xlsx'), 
                                     sheet_name='platelet', engine='openpyxl')['Gene names'].tolist()
df_loadings['platelet_marker'] = np.where(df_loadings['Gene name'].isin(qc_markers_platelet), 1, 0)

In [ ]:
df_toplot = data_plasma_filtered_log.T.join(data_cli_prot[['visit_date_year']], how='left')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3))
labels = ['a', 'b', 'c']
for ax, label in zip(axs, labels):
    # Arguments are x, y (relative to the axes), text, and any text properties
    ax.text(-0.3, 1.1, label, transform=ax.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

generate_pca_plot1(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, 
                            group_column='time_to_analysis', palette='bwr', ax=axs[0])
legend = axs[0].get_legend()
legend.set_title('time to\nanalysis')

sns.scatterplot(data=df_loadings, x='PC1', y='PC2',hue='platelet_marker', ax=axs[1])
sns.boxplot(y='Q9Y490_TLN1', x='visit_date_year',
            data=df_toplot, width=0.5, color=colors[1], ax=axs[2])
total_ticks = len(axs[2].get_xticklabels())
new_labels = [''] * total_ticks
indices_to_label = [1, total_ticks // 2, total_ticks - 1]
for i in indices_to_label:
    new_labels[i] = axs[2].get_xticklabels()[i].get_text()
axs[2].set_xticklabels(new_labels)
axs[2].set_xlabel('Year of sampling')
axs[2].set_ylabel('Q9Y490_TLN1\nIntensity [Log2]')

plt.subplots_adjust(wspace=0.8)
axs[1].set_title('Loading plot')
axs[0].set_title('PCA plot')
for ax in axs:
    ax.set_xlabel(ax.get_xlabel(), fontsize=12)
    ax.set_ylabel(ax.get_ylabel(), fontsize=12)
    ax.tick_params(axis='x', labelsize=12)
    ax.tick_params(axis='y', labelsize=12)
plt.rcParams['pdf.fonttype'] = 42    
#fig.savefig(os.path.join(folders['figures'], '1.png'), bbox_inches='tight', dpi=120)
fig.savefig('figures/platelet.pdf')

#### Imputation

In [ ]:
data_plasma_filtered_log_imputed = data_plasma_filtered_log.apply(imputation_normal_distribution)
value = data_plasma_filtered_log_imputed.loc['Q9Y696_CLIC4', 'Plate10_1']
assert abs(value - 3.149335) < 0.00001, 'Imputed value changed in comparison to previous run'

In [ ]:
X_train = data_plasma_filtered_log_imputed.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2)
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

#### Normalization

In [ ]:
ids_2k = annotation_file[annotation_file['Cohort']=='holbaek_2k']['Sample ID']
ids_1k = annotation_file[annotation_file['Cohort']=='holbaek_1k']['Sample ID']
ids_2k = list(set(ids_2k) & set(data_plasma_filtered.columns))
ids_1k = list(set(ids_1k) & set(data_plasma_filtered.columns))

In [ ]:
IDmapping_sampleID_to_Batch = dict(zip(annotation_file['Sample ID'], annotation_file['Grouping_batch']))
IDmapping_sampleID_to_Instrument = dict(zip(annotation_file['Sample ID'], annotation_file['Instrument']))

In [ ]:
RE_COMBAT = False
if RE_COMBAT:
    from combat.pycombat import pycombat
    dm = data_plasma_filtered_log_imputed[sample_ids].copy()
    batch_sample_prep = [IDmapping_sampleID_to_Batch[i] for i in dm.columns]
    batch_instrument = [IDmapping_sampleID_to_Instrument[i] for i in dm.columns]
    dm1 = pycombat(dm, batch_sample_prep)
    dm2 = pycombat(dm1, batch_instrument)
    data_plasma_filtered_log_imputed_corrected = dm2.copy()
    dm2.to_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected.csv')
else:
    data_plasma_filtered_log_imputed_corrected = pd.read_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected.csv').set_index('ProteinID_Genename')
    data_plasma_filtered_log_imputed_corrected = data_plasma_filtered_log_imputed_corrected.rename_axis('Sample ID', axis=1)

value = data_plasma_filtered_log_imputed_corrected.loc['Q9Y6Z7_COLEC10', 'Plate1_2']
assert abs(value - 9.070180) < 0.00001, 'Corrected value changed in comparison to previous run'

#### Dataset w normalization w/o imputation

In [ ]:
mask = data_plasma_filtered_log.isna()
data_reverted = data_plasma_filtered_log_imputed_corrected.mask(mask)

In [ ]:
X_train = data_plasma_filtered_log_imputed_corrected.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2)
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

#### Save dataset

In [ ]:
double_key = data_cli_prot[['blood_sample_number']]
annotation_file_export = annotation_file.set_index('Sample ID').join(double_key, how='right')

In [ ]:
def format_dataset(df):
    df_new=df.copy()
    df_new.rename_axis('ProteinID_Genename', axis=0, inplace=True)
    cols_to_add = ['Protein ID', 'Protein IDs', 'Gene name', 'Gene names']
    df_new=df_new.join(protein_ids.set_index('ProteinID_Genename')[cols_to_add])
    return df_new

# FILE_RESULTS = os.path.join(DATA_FOLDER_PROCESSED, 'proteomics_datasets.xlsx')
# with pd.ExcelWriter(FILE_RESULTS) as writer:
#      format_dataset(data_plasma_raw).to_excel(writer, sheet_name='raw')
#      format_dataset(data_plasma_filtered_log).to_excel(writer, sheet_name='filtered_log2')
#      format_dataset(data_plasma_filtered_log_imputed).to_excel(writer, sheet_name='filtered_log2_imputed')
#      format_dataset(data_plasma_filtered_log_imputed_corrected).to_excel(writer, sheet_name='imputed_batchcorrected')
#      format_dataset(data_reverted).to_excel(writer, sheet_name='batchcorrected_no_imputation')
#      annotation_file_export.to_excel(writer, sheet_name='annotation_file')

### Quality assessment

#### Calculate CV based on 94 quality assessment samples (pooled plasma allocated in 24 plates)

In [ ]:
qa_plasma = annotation_file[annotation_file['Grouping_batch'] == '2k_QA']['Sample ID'].tolist()
df_cv = calculate_cv(data_plasma_filtered, qa_samples=qa_plasma).sort_values(by='Coefficient of variation')
df_cv['Gene name'] = df_cv.index.str.split('_').str[1]

In [ ]:
folders['results']

In [ ]:
df_cv.to_csv(os.path.join(folders['results'], 'df_cv.csv'))

#### Depth

In [ ]:
prot_dep_wide = pd.DataFrame({'raw': data_plasma_raw.count(), 'filtered':data_plasma_filtered.count()})
prot_dep = pd.melt(prot_dep_wide, var_name='dataset', value_name='Number of proteins')
prot_dep.groupby('dataset')['Number of proteins'].median()

#### Supplementary figure 1

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(11,3))
sns.boxplot(data = prot_dep, x='dataset', y='Number of proteins', color='white', ax=ax1)
#ax1.set_ylim(0, 520)
for i,box in enumerate(ax1.artists):
    box.set_edgecolor('black')
    box.set_facecolor('white')

    # iterate over whiskers and median lines
    for j in range(6*i,6*(i+1)):
         ax1.lines[j].set_color('black')
ax2 = sns.scatterplot(data=df_cv, x='rank', y='Protein abundance [Log10]', ax=ax2)
ax3 = sns.scatterplot(data=df_cv, x='Protein abundance [Log2]', y='Coefficient of variation', hue='color', ax=ax3, size=5)
plt.rcParams['pdf.fonttype'] = 42
plt.savefig('figures/data_quality.pdf', dpi=120, bbox_inches='tight')

### Export data for Dash app

In [ ]:
data_dash = data_reverted.T.join(data_cli_prot).rename_axis('Sample ID', axis=0)[proteins + ['age_int', 'sex', 'z_BMI.Nysom']]
Path(os.path.join(folders['dash'], 'dataset')).mkdir(exist_ok=True)
data_dash.to_csv(os.path.join(folders['dash'], 'dataset/data_age_sex.csv'))

### Combine proteomics and clinical data

In [ ]:
data_combined = data_plasma_filtered_log_imputed_corrected.T.join(data_cli_prot).rename_axis('Sample ID', axis=0)
data_combined = data_combined.join(annotation_file.set_index('Sample ID'), how='left')

#### Check sample quality markers by sample collection time

In [ ]:
REPLOT_FIGURES = False
Path(os.path.join(folders['figures'], 'quality_markers')).mkdir(exist_ok=True)
proteins_to_plot = protein_ids[protein_ids['Gene name'].isin(qc_markers_platelet)]['ProteinID_Genename'].tolist()
proteins_to_plot = set(proteins_to_plot) & set(data_combined.columns)
if REPLOT_FIGURES:
    for i in proteins_to_plot:
        fig, ax = plt.subplots()
        sns.boxplot(x='visit_date_year', y=i, data=data_combined, color='crimson', )
        plt.title(i, fontsize=16)
        plt.xticks(rotation=45)
        plt.savefig(os.path.join(folders['figures'], 'quality_markers/{}'.format(i)), bbox_inches='tight', dpi=120)
        plt.close(fig)

#### Revision 

In [ ]:
data_combined_noimpute = data_reverted.T.join(data_cli_prot).rename_axis('Sample ID', axis=0)
data_combined_noimpute = data_combined_noimpute.join(annotation_file.set_index('Sample ID'), how='left')

In [ ]:
data_combined_noimpute['hs_CRP_SSI_log2']=data_combined_noimpute['hs_CRP_SSI'].apply(np.log2)

In [ ]:
data_combined_noimpute[['O43866_CD5L', 'P02741_CRP', 'hs_CRP_SSI_log2', 'P01871_IGHM', 'P01591_JCHAIN']].corr()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))  # 1 row, 2 columns

# Define the columns for plotting
cols = [('O43866_CD5L', 'P01871_IGHM'), ('O43866_CD5L', 'P01591_JCHAIN')]
#cols = [('O43866_CD5L', 'P02741_CRP'), ('P01871_IGHM', 'P02741_CRP')]
for i, (x, y) in enumerate(cols):
    # Plot
    sns.scatterplot(ax=axes[i], x=x, y=y, data=data_combined_noimpute)
    axes[i].set_title(f'{x} vs {y}')
    
    axes[i].set_xlabel(f'{x} [Log2]')
    axes[i].set_ylabel(f'{y} [Log2]')

    # Calculate and annotate Pearson correlation
    df_test = data_combined_noimpute[[x, y]].dropna()
    corr = np.corrcoef(df_test[x], df_test[y])[0, 1]
    #corr = np.corrcoef(data_combined_noimpute[x].dropna(), data_combined_noimpute[y].dropna())[0, 1]
    axes[i].text(0.05, 0.95, f'Pearsonr: {corr:.2f}', transform=axes[i].transAxes, 
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
Path(os.path.join(folders['figures'], 'revision')).mkdir(exist_ok=True)
plt.savefig(os.path.join(folders['figures'], 'revision/IGHM.png'), bbox_inches='tight', dpi=120)
plt.show()

### Data exploration

#### Normality test before transformation

In [ ]:
# Long data format to ease computation 
data_long = data_combined[proteins].melt(var_name='ProteinID_Genename', value_name='MS signal [Log2]').set_index('ProteinID_Genename')

from src.statistical_testing import normality_pg
# Set a dummy variable needed for pingouin.normality
data_long['group_dummy']=1
normality_results = normality_pg(data=data_long, dv='MS signal [Log2]', group='group_dummy')

# Show results
print('Number of protein with non-normal distribution:{}'.format(normality_results['normal'].value_counts()[False]))

#### Ranked-based inverse normalized transformation (INT)
- https://stackoverflow.com/questions/15549836/transform-data-to-fit-normal-distribution
- [good discussion on violating OLS residual normality assumption](https://stats.stackexchange.com/questions/29731/regression-when-the-ols-residuals-are-not-normally-distributed)
- [good discussion on testing non-linear association](https://stats.stackexchange.com/questions/35893/how-do-i-test-a-nonlinear-association)

In [ ]:
# Perform ranked-based inverse normalized transformation (INT) on protein levels per protein
# Refer to https://github.com/edm1/rank-based-INT

from src.rank_based_int import rank_INT
RE_INT = False
data_2k = data_plasma_filtered_log_imputed_corrected[ids_2k]
data_1k = data_plasma_filtered_log_imputed_corrected[ids_1k]

if not RE_INT:
    data_proteomics_int = pd.read_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected_int.csv').set_index('Sample ID')
else:
    new_df = []
    for protein in tqdm(proteins):
        new_df.append(pd.DataFrame(rank_INT(data_2k.T[protein], stochastic=False), columns=[protein]))
    data_proteomics_int = pd.concat(new_df, axis=1)
    data_proteomics_int.rename_axis('Sample ID', axis=0, inplace=True)
    data_proteomics_int.to_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected_int.csv')

In [ ]:
value = data_proteomics_int.loc['Plate1_2', 'Q9Y6Z7_COLEC10']
assert abs(value - 0.710323) < 0.00001, 'Normalized value changed in comparison to previous run'

In [ ]:
data_combined_int = data_proteomics_int.join(data_cli_prot).rename_axis('Sample ID', axis=0)
data_combined_int = data_combined_int.join(df_pc['PC1'], how='left')

#### Multivariate normality test after data transformation

In [ ]:
covariates = ['sex', 'z_BMI.Nysom' , 'age', 'time_to_analysis', 'pubertal_status2', ]
lr_residuals = {}
for protein in tqdm(proteins):
    lr = pg.linear_regression(X=data_combined_int[covariates], y=data_combined_int[protein], remove_na=True)
    residuals = lr.residuals_    
    lr_residuals[protein]=residuals

In [ ]:
normality_test_residuals = []
for protein in lr_residuals.keys():
    normal = pg.normality(lr_residuals[protein])
    normal['protein']=protein
    normality_test_residuals.append(normal)
pd.concat(normality_test_residuals)['normal'].value_counts()

#### Multicolinearity test after data transformation

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [ ]:
vif_dm = data_combined_int[covariates_lm].dropna()
vif_dm = add_constant(vif_dm)
#vif_dm['constant']=1
vif_data = pd.DataFrame()
vif_data["feature"] = vif_dm.columns 
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(vif_dm.values, i) for i in range(len(vif_dm.columns))] 
print(vif_data)

#### Investigate PC1 and PC2 after batch correction

In [ ]:
X_train = data_plasma_filtered_log_imputed_corrected.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
df_pc = df_pc.join(data_cli_prot[covariates + ['obesity', 'IgG']])
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=3, group_column='obesity')
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

In [ ]:
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, group_column='time_to_analysis', palette='bwr')

### Protein-phenotype associations

#### Run linear regression adjusting for covariates

In [ ]:
# overweight/obesity (BMI SDS >= 1.28)
data_combined_int['overweight'] = np.where(data_combined_int['z_BMI.Nysom']>=1.28, 1, 0)
data_combined_int['overweight*z_BMI.Nysom']=data_combined_int['overweight']*data_combined_int['z_BMI.Nysom']

In [ ]:
# Save for proportion of variance explained estimation
#data_combined_int['Participant ID']=data_combined_int.index.map(IDmapping_SampleID_to_bloodSampleID)
#data_combined_int.to_csv(os.path.join(DATA_FOLDER_PROCESSED, 'data_combined_int.csv'))

In [ ]:
Path(os.path.join(folders['results'], 'association')).mkdir(exist_ok=True)

In [ ]:
from src.statistical_testing import perform_linear_regression

In [ ]:
# pubertal_status2 (1=prepubertal, 2=pubertal/postpubertal), time_to_analysis (sample storage time), z_BMI.Nysom (BMI SDS, beta estimated seperately for normal and overweight (z_BMI.Nysom >=1.28))
covariates_lm =['age', 'sex', 'z_BMI.Nysom','overweight*z_BMI.Nysom' , 'overweight',
                'time_to_analysis', 'pubertal_status2', 'PC1', ]

results_path = Path(folders['results'], 'association')
RE_LIREG = False

if not RE_LIREG:
    stats_lr = pd.read_csv(results_path / 'lireg_statistics.csv')
    residuals_lr = pd.read_csv(results_path / 'lireg_residuals.csv').set_index('Sample ID')
else:
    stats_lr, residuals_lr = perform_linear_regression(data_combined_int, proteins, covariates_lm)
    residuals_lr=pd.DataFrame.from_dict(residuals_lr).rename_axis('Sample ID', axis=0)
    # Save results
    stats_lr.to_csv(results_path / 'lireg_statistics.csv', index=False)   
    residuals_lr.to_csv(results_path / 'lireg_residuals.csv')

In [ ]:
value = residuals_lr.loc['Plate5_1', 'Q9Y6Z7_COLEC10']
assert abs(value - -1.214874) < 0.00001, 'Regressed value changed in comparison to previous run'

In [ ]:
normality_test_residuals = []
for protein in residuals_lr.columns:
    normal = pg.normality(residuals_lr[protein])
    normal['protein']=protein
    normality_test_residuals.append(normal)
pd.concat(normality_test_residuals)['normal'].value_counts()

In [ ]:
dict_BMI = {'overweight*z_BMI.Nysom':'z_BMI.Nysom', }
stats_lr['names2'] = stats_lr['names'].replace(dict_BMI)

In [ ]:
stats_lr['Gene name']=stats_lr['dep_var'].str.split('_').str[1]

In [ ]:
stats_lr['direction2']=stats_lr['direction']+'_'+ stats_lr['names']
stats_lr['direction2']=stats_lr['direction2'].replace({i:'not significant' for i in stats_lr['direction2'] if i.startswith('not sig')})

In [ ]:
sig_stats=stats_lr[stats_lr.rejected]
sig_stats['names'].value_counts()/len(proteins)

In [ ]:
from scipy import stats
aaaa = ['P01833_PIGR', 'Q92954_PRG4', 'P05019_IGF1', 'P04278_SHBG', 'Q13790_APOF']
protein = 'Q92954_PRG4'
fig, ax=plt.subplots(figsize=(3,3))
df=data_combined.dropna(subset=['z_BMI.Nysom', protein])
values = np.vstack([df['z_BMI.Nysom'], df[protein]])
kernel = stats.gaussian_kde(values)(values)
sns.scatterplot(x='z_BMI.Nysom', y=protein, data=df, c=kernel, cmap='viridis')
#sns.lmplot(x='z_BMI.Nysom', y=protein, data=df, hue='obesity')

In [ ]:
df=data_combined_int.dropna(subset=['z_BMI.Nysom', protein])
values = np.vstack([df['z_BMI.Nysom'], df[protein]])
kernel = stats.gaussian_kde(values)(values)
sns.scatterplot(x='z_BMI.Nysom', y=protein, data=df, c=kernel, cmap='viridis')
sns.lmplot(x='z_BMI.Nysom', y=protein, data=df, hue='obesity')

#### Associated with age, sex and BMI-SDS

In [ ]:
# Extract proteins associated with three factors
factors_tokeep = ['sex', 'z_BMI.Nysom', 'age',]
sig_stats[sig_stats['names2'].isin(factors_tokeep)]['dep_var'].nunique()
sig_stats['names2'].value_counts()/len(proteins)

In [ ]:
# Create a dictionary to ease plotting
lireg_dicts = {}
for covariate in ['age', 'sex','z_BMI.Nysom']:
    lireg_dicts[covariate] = {'df':stats_lr[stats_lr['names2']==covariate], 
                         'sig':sig_stats[sig_stats['names2']==covariate],
                        'sig_set':set(sig_stats[sig_stats['names2']==covariate]['dep_var'])}

In [ ]:
# Explore associations
FACTORS = ['age', 'z_BMI.Nysom', 'sex',]
dfs=[lireg_dicts[i]['df'] for i in FACTORS]
data_frame=lireg_dicts['sex']['df']
px.scatter(x='coef', y='-Log10 P-value', data_frame=data_frame, hover_name='dep_var')

In [ ]:
igfs = lireg_dicts['age']['df'][lireg_dicts['age']['df']['Gene name'].str.startswith('IGF')]
igfs = igfs[igfs.rejected]

### Reviewer question
- which proteins appear to be exclusive to obesity

In [ ]:
df=lireg_dicts['age']['sig']
df['Gene name']=df['dep_var'].str.split('_').str[1]
df=df.assign(abs_coef = abs(df['coef']))
list_tocheck = df[df['abs_coef']>0.06]['Gene name'].tolist()

In [ ]:
import gseapy as gp

In [ ]:
gp.get_library_name()
pathways = ['MSigDB_Hallmark_2020',
            'GO_Biological_Process_2023', 
            'KEGG_2021_Human',
            'GO_Molecular_Function_2023']

In [ ]:
background_genes=data_plasma_filtered.index.str.split('_').str[1].tolist()

In [ ]:
enr_proteome = gp.enrichr(gene_list=background_genes, # or "./tests/data/gene_list.txt",
                 gene_sets=pathways,
                 organism='human', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir=None, # don't write to disk
                )

In [ ]:
results_full = enr_proteome.results
results_full['Nr.proteins']=results_full['Overlap'].str.split('/').str[0].astype(int)
sig_full = results_full[results_full['Adjusted P-value']<0.05]

In [ ]:
sig_full = results_full[results_full['Adjusted P-value']<0.05]

In [ ]:
sig_full['Adjusted P-value [-Log10]']=-np.log10(sig_full['Adjusted P-value'])

In [ ]:
tocheck=[i for i in sig_full['Term'] if 'Epidermal' in i]

In [ ]:
terms_toinclude = ['heme Metabolism',
                   'PI3K-Akt signaling pathway','Estrogen signaling pathway',
                   'mTORC1 Signaling',
                   'Regulation Of Epidermal Growth Factor Receptor Signaling Pathway (GO:0042058)',
                   'Complement and coagulation cascades', 
                   'Proteolysis (GO:0006508)', 'ECM-receptor interaction'
                   'Glycolysis', 'Cell adhesion molecules', 'Protein Transport (GO:0015031)',
                   'Inflammatory Response (GO:0006954)',
                   'Regulation Of Immune Response (GO:0050776)',
                   'Angiogenesis',
                   'Xenobiotic Metabolism',
                   'Regulation Of Endopeptidase Activity (GO:0052548)',
                   'Apoptosis',
                   'Response To Steroid Hormone (GO:0048545)',
                   'Fatty Acid Metabolism',
                   'Triglyceride Homeostasis (GO:0070328)',
                   'Glucose Metabolic Process (GO:0006006)',               
                   'Skeletal System Development (GO:0001501)', 'Adipogenesis', 'Cholesterol metabolism'
                  ]

In [ ]:
fig, ax = plt.subplots(figsize=(1, 4))
df_to_plot = sig_full[sig_full['Term'].isin(terms_toinclude)].sort_values(by='Nr.proteins')
sns.barplot(x='Nr.proteins', y='Term', data=df_to_plot, color=colors[1])
plt.rcParams['pdf.fonttype'] = 42
plt.savefig('figures/figure1_2.pdf', dpi=120, bbox_inches='tight')

#### SourceData Fig.2 

In [ ]:
sourcedata_fig2a = df_to_plot[['Gene_set','Nr.proteins', 
                               'Term', 'Overlap', 'Genes']].sort_values(by='Nr.proteins', ascending=False)
sourcedata_fig2a.reset_index(drop=True, inplace=True)

In [ ]:
sourcedata_fig2a.to_excel('source_data/SourceData_Figure2.xlsx')

#### Reviewer question
- Relationship between age- and SNP-associated proteins

In [ ]:
pqtl = pd.read_csv('final_set.csv')

In [ ]:
from src.utils import intersection

### By tissue

In [ ]:
hpa = pd.read_csv(os.path.join(folders['annotations'], 'proteinatlas.tsv'), sep='\t')

In [ ]:
cols_tokeep = ['Gene', 'Uniprot', 'Protein class', 'RNA tissue specificity', 'RNA tissue specific nTPM']
df_hpa = hpa[cols_tokeep]
df_hpa = df_hpa.drop_duplicates(subset='Gene', keep='first')
df_tissue = df_cv.merge(df_hpa, left_on='Gene name', right_on='Gene', how='left')

In [ ]:
df_tissue['RNA tissue specificity'].value_counts()

In [ ]:
df_tissue['RNA tissue specificity'].value_counts(1)

In [ ]:
tissues_expanded = df_tissue['RNA tissue specific nTPM'].str.split(';', expand=True).stack().reset_index(level=1, drop=True)
tissues_expanded.name = 'Tissue_nTPM'
expanded_data = df_tissue.join(tissues_expanded)
expanded_data['Tissue'] = expanded_data['Tissue_nTPM'].str.extract(r'(^.*?):')
tissue_counts_by_specificity = expanded_data.groupby(['RNA tissue specificity', 'Tissue']).size().unstack(fill_value=0)

In [ ]:
expanded_data['enriched_tissue']=expanded_data['RNA tissue specific nTPM'].str.split(':').str[0]

In [ ]:
age_genes = [i.split('_')[1] for i in lireg_dicts['age']['sig_set']]
sex_genes = [i.split('_')[1] for i in lireg_dicts['sex']['sig_set']]
bmi_genes = [i.split('_')[1] for i in lireg_dicts['z_BMI.Nysom']['sig_set']]

In [ ]:
expanded_data_enr = expanded_data[expanded_data['RNA tissue specificity'].isin(['Tissue enriched', 'Group enriched'])]

In [ ]:
expanded_data[expanded_data['Gene name'].isin(bmi_genes)].head(2)

In [ ]:
# Sum the occurrences across all three categories for each tissue type
summed_counts = tissue_counts_by_specificity.sum(axis=0).sort_values(ascending=False)
to_drop = summed_counts[summed_counts < 15].index

# Prepare data for plotting
data_to_plot = tissue_counts_by_specificity.reindex(columns=summed_counts.index)
new_order = ['Tissue enriched', 'Tissue enhanced', 'Group enriched']
data_to_plot = data_to_plot.loc[new_order]
data_to_plot = data_to_plot.drop(to_drop, axis=1)

# Plotting
fig, ax = plt.subplots(figsize=(1, 4))
data_to_plot.T.plot(kind='barh', stacked=True, ax=ax, color=colors[:3])
ax.set_title('Summed Number of Tissues Across RNA Tissue Specificity Categories')
ax.set_xlabel('Number of Proteins')
ax.set_ylabel('Number of Occurrences')
ax.legend(title='RNA Tissue Specificity')
ax.invert_yaxis()

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.rcParams['pdf.fonttype'] = 42
plt.savefig('figures/figure1_1.pdf', dpi=120, bbox_inches='tight')

In [ ]:
df_try = lireg_dicts['z_BMI.Nysom']['df']
df_try['Gene name']=df_try['dep_var'].str.split('_').str[1]
df_try=df_try.merge(df_hpa, left_on='Gene name', right_on='Gene', how='left')
df_try[df_try.rejected].sort_values(by='coef', ascending=True)[:5]

#### Table S

In [ ]:
cols_tokeep = ['Protein ID', 'Gene name', 'names', 'coef', 'se', 'T', 'pval', 'df_model', 'df_residual',
               'r2', 'adj_r2', 'CI[2.5%]', 'CI[97.5%]',
               'nr_obs', 'qvalue', '-Log10 P-value', ]
headers_toreplace = ['Protein ID', 'Gene name', 'variable', 'coefficient', 'standard error', 'T-value', 'p-value', 
                    'degree of freedom_model', 'degree of freedom_residual', 'R square', 'adjusted R square', 'CI[2.5%]', 'CI[97.5%]',
                    'observations', 'BH-corrected p-value', '-Log10 p-value']

In [ ]:
df = sig_stats.copy()
var_tokeep = ['age', 'sex', 'overweight*z_BMI.Nysom', 'overweight', 'z_BMI.Nysom']
col_todrop = ['rejected', 'direction']

In [ ]:
df = df[df['names'].isin(var_tokeep)].drop(col_todrop, axis=1).sort_values(by='names')
df = df.reset_index().drop(['index'], axis=1)
df['Protein ID'] = df['dep_var'].str.split('_').str[0]
df['Gene name'] = df['dep_var'].str.split('_').str[1]
df = df.drop(['dep_var'], axis=1)
# Replace column headers
new_names = dict(zip(cols_tokeep, headers_toreplace))
df = df[cols_tokeep].rename(new_names, axis=1).sort_values(by=['variable', 'R square'], ascending=False)
df = df.merge(df_tissue[['Gene name', 'Protein class', 'RNA tissue specificity', 'RNA tissue specific nTPM']], on='Gene name', how='left')
df_table2_formatted=df.copy()

### Reviewer question - interaction effects between age, sex and BMI on protein abundance

In [ ]:
df = data_combined_int.copy()
c1, c2, c3 = 'age', 'sex', 'z_BMI.Nysom'
df['{}*{}'.format(c1, c2)] = df[c1] * df[c2]
df['{}*{}'.format(c1, c3)] = df[c1] * df[c3]
df['{}*{}'.format(c2, c3)] = df[c2] * df[c3]
df['{}*{}*{}'.format(c1, c2, c3)] = df[c1] * df[c2] *df[c3]
df_interact = df.copy()
interaction_terms = ['age*sex', 'age*z_BMI.Nysom', 'sex*z_BMI.Nysom', 'age*sex*z_BMI.Nysom']
covariates_interaction = covariates_lm + interaction_terms

In [ ]:
df_interact.to_csv(DATA_FOLDER_PROCESSED / 'data_combine_int_interact.csv', sep='\t')

In [ ]:
# linear regression done in Computerome 
stats_interact = pd.read_csv(folders['results'] / 'association/lireg_all_interaction_combined.stats', sep='\t')

In [ ]:
# Extract significant associations
sig_stats_interact = stats_interact[stats_interact.rejected]

# Extract proteins associated with three factors
factors_tokeep_interact = FACTORS + interaction_terms
sig_stats_interact[sig_stats_interact['names'].isin(factors_tokeep_interact)]['dep_var'].nunique()
sig_stats_interact['names'].value_counts()

In [ ]:
df = sig_stats_interact.copy()
col_todrop = ['rejected', 'direction']

In [ ]:
df = df[df['names'].isin(factors_tokeep_interact)].drop(col_todrop, axis=1).sort_values(by='names')
df = df.reset_index().drop(['index'], axis=1)
df['Protein ID'] = df['dep_var'].str.split('_').str[0]
df['Gene name'] = df['dep_var'].str.split('_').str[1]
df = df.drop(['dep_var'], axis=1)
df = df[cols_tokeep].rename(new_names, axis=1).sort_values(by=['variable', 'R square'], ascending=False)
df=df.merge(df_tissue[['Gene name', 'Protein class', 'RNA tissue specificity', 'RNA tissue specific nTPM']], on='Gene name', how='left')
df_table3_formatted = df.copy()

#### Export data for Dash app

In [ ]:
df_dash = stats_lr.copy()
df_dash = df_dash[df_dash['names'].isin(['age', 'sex', 'overweight*z_BMI.Nysom', 'overweight', 'z_BMI.Nysom'])]
df_dash = df_dash.rename(dict(zip(cols_tokeep, headers_toreplace)), axis=1)
df_dash['p-value [-Log10]']=-np.log10(df_dash['p-value'])
df_dash['BH-corrected p-value [-Log10]']=-np.log10(df_dash['BH-corrected p-value'])

df_dash = df_dash[['dep_var', 'variable', 'coefficient', 'standard error','observations', 'p-value [-Log10]', 'BH-corrected p-value [-Log10]', 'rejected']]
df_dash=df_dash.rename({'dep_var':'ProteinID_Genename', 'variable':'factor', 'rejected':'significant'}, axis=1)
df_dash.to_csv(folders['dash'] / 'dataset/dataset2.csv', index=False)

#### Characteristics of participants included in the phenotype-protein association analysis

In [ ]:
data_combined_int[['sex', 'age_int', 'z_BMI.Nysom', 'BMI', 'pubertal_status2']].groupby('sex')['pubertal_status2'].value_counts()

### Data processing for protein-genotype association
- Including puberty stage as a covariate results in losing >500 samples due to incomplete data. 
- In this step, we decided not to include puberty stage. 

In [ ]:
# Define the list of covariates for pqtl analysis
covariates_pqtl = ['age', 'sex', 'z_BMI.Nysom', 'overweight*z_BMI.Nysom', 
                   'overweight', 'time_to_analysis', 'PC1']

In [ ]:
RE_LIREG = False

if not RE_LIREG:
    stats_pqtl = pd.read_csv(gemma_path / 'lireg_statistics.csv')
    residuals_pqtl = pd.read_csv(gemma_path / 'lireg_residuals.csv').set_index('Sample ID')
else:
    stats_pqtl, residuals_pqtl = perform_linear_regression(data_combined_int, proteins, covariates_pqtl)
    # Save results
    stats_pqtl.to_csv(gemma_path / 'lireg_statistics.csv', index=False)
    residuals_pqtl = pd.DataFrame.from_dict(residuals_pqtl).rename_axis('Sample ID', axis=0)
    residuals_pqtl.to_csv(gemma_path / 'lireg_residuals.csv')

In [ ]:
value=residuals_pqtl.loc['Plate5_1', 'Q9Y6Z7_COLEC10']
assert abs(value - -1.209057) < 1e-4, 'Regressed value for GWAS changed in comparison to previous run'

#### Perform INT on residuals

In [ ]:
RE_INT = False

if not RE_INT:
    data_gwas_int = pd.read_pickle(gemma_path / 'lireg_residuals_int.pkl')
else:
    new_df = []
    data = residuals_pqtl
    for protein in tqdm(data.columns):
        new_df.append(pd.DataFrame(rank_INT(data[protein], stochastic=False), columns=[protein]))
    data_gwas_int = pd.concat(new_df, axis=1)
    data_gwas_int.rename_axis('Sample ID', axis=0, inplace=True)
    data_gwas_int.to_pickle(gemma_path / 'lireg_residuals_int.pkl')

In [ ]:
value=data_gwas_int.loc['Plate5_1', 'Q9Y6Z7_COLEC10']
assert abs(value - -1.706134) < 1e-4, 'Normalized value for GWAS changed in comparison to previous run'

In [ ]:
X_train = data_gwas_int.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
df_pc = df_pc.join(data_cli_prot[covariates + ['obesity', 'IgA', 'IgG', 'IgM']])
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, group_column='obesity', palette='bwr')

#### Prepare the data to fit the GEMMA input format

In [ ]:
# Replace sample ID with participant ID so it's compatible with genotype data
participant_ids = '66-' + data_gwas_int.index.map(IDmapping_SampleID_to_bloodSampleID)
data_gwas_int.insert(0, 'Participant ID', participant_ids)

In [ ]:
Path(os.path.join(folders['pQTL'], 'QC_PLINK')).mkdir(exist_ok=True)

In [ ]:
# Import the .fam file template
fam_tep = pd.read_csv(folders['pQTL'] / 'QC_PLINK/target5.fam', header=None, sep=' ')
data_gwas_export = fam_tep.set_index(0).join(data_gwas_int.round(5).set_index('Participant ID')).reset_index().drop([5], axis=1)
data_gwas_export.fillna('NA', inplace=True)

# Export data for GWAS
data_gwas_export.to_csv(gemma_path / 'export/phenotype.fam', header=None, index=False, sep=' ')

In [ ]:
# Check if data changed in comparison to previous run
value = data_gwas_export.loc[0, 'P01009_SERPINA1']
assert abs(value - 1.50934) < 1e-4, 'Exported value for GWAS changed in comparison to previous run'

In [ ]:
# Create a dataframe mapping phenotype IDs to protein IDs
nr_proteins = len(proteins)
proteinID_pqtl=pd.DataFrame({'Phenotype ID':np.arange(1, nr_proteins+1),'Protein ID':data_gwas_export.columns[5:]})
proteinID_pqtl.to_csv(gemma_path / 'export/ProteinID.txt', index=False, sep='\t')

with open(gemma_path / 'export/phenotype.list', 'w') as file:
    for line in np.arange(1, nr_proteins+2):
        file.write(str(line) + '\n')
        
# Association performed in Computerome (GEMMA v0.98.3)

#### Export data to look at "dose-response"

In [ ]:
# Define paths
dose_resp_path = folders['pQTL'] / 'dose-response'
dose_resp_path.mkdir(exist_ok=True)
excel_path = dose_resp_path / 'dose_response.xlsx'

# Define a function to prepare dataframes
def prepare_df(df, col_prefix, index_col=None):
    df = df.copy()
    if index_col:
        df[index_col] = col_prefix + df.index.map(IDmapping_SampleID_to_bloodSampleID)
        df = df.set_index(index_col)
    else:
        df.columns = col_prefix + df.columns.map(IDmapping_SampleID_to_bloodSampleID)
    return df

#Writing to Excel with prepared dataframes
with pd.ExcelWriter(excel_path) as writer:
    prepare_df(data_plasma_filtered_log_imputed_corrected, '66-').to_excel(writer, sheet_name='data_log2_imputed')
    prepare_df(data_reverted, '66-').to_excel(writer, sheet_name='data_log2')
    prepare_df(data_cli_prot[['sex', 'age_year', 'obesity', 'height']].dropna(), '66-', 'Participant ID').to_excel(writer, sheet_name='data_cli')

In [ ]:
protein_ids[protein_ids['ProteinID_Genename'].isin(proteins)][['Protein ID', 'Gene name']].to_csv('aa.csv')

#### Export sample ID annotation for peptide data inspection

In [ ]:
df_annotation_pep = annotation_file.copy()
df_annotation_pep['Participant ID'] = '66-' + df_annotation_pep['Sample ID'].map(IDmapping_SampleID_to_bloodSampleID)
df_annotation_pep.to_csv(folders['pQTL'] / 'peptide_annotation.csv', index=False)

# Peptide-level inspection was performed in Computerome

#### Prepare genotype array batch data for use as covariates ####

In [ ]:
# import batch info
df_batch = pd.read_csv(folders['pQTL'] / 'ids_in_batch.txt', sep=' ', header=None, names=['participant ID', 'batch'])
onehot_batch = pd.DataFrame(pd.get_dummies(df_batch['batch']))
onehot_batch['participant ID']=df_batch['participant ID']

In [ ]:
# Combine participant IDs and genotype array batch data as covariates
data_gwas_covariates = fam_tep[[0]].set_index(0).join(onehot_batch.set_index('participant ID')).drop(['batch2015'], axis=1)
# Add an intercept column to the covariate data
data_gwas_covariates.insert(0, 'intercept', 1)
# Replace missing values with 'NA'
data_gwas_covariates = data_gwas_covariates.fillna('NA')
# Save the covariate data to a tab-separated text file for use with GEMMA
data_gwas_covariates.to_csv(gemma_path / 'export/covariates.txt', header=None, index=False, sep=' ')

#### Check sex discrepancy

In [ ]:
# Read in the PLINK sexcheck file and merge with clinical data
df_sex = pd.read_csv(folders['pQTL'] / 'QC_PLINK/plink.sexcheck', sep='\s+')
df_sex['biobank_ID']=df_sex['FID'].str.split('-').str[1]
df_sex = df_sex.merge(data_cli_raw[['biobank_ID', 'sex']], on='biobank_ID', how='left')

# Select rows where the SNPSEX column does not match the sex column from clinical data
df_sex[df_sex['SNPSEX']!=df_sex['sex']]

### Can we predict age and BMI based on plasma proteome? 

In [ ]:
features_age, df_features_age = get_features(outcome_col='age', predictors=proteins, stratify_col='age_int', data=data_combined, top=50)
features_bmi, df_features_bmi = get_features(outcome_col='BMI', predictors=proteins, stratify_col = 'bin_numbers_bmi', data=data_combined, top=50)

In [ ]:
features_bmisds, df_features_bmisds = get_features(outcome_col='z_BMI.Nysom', 
                                                   predictors=proteins, stratify_col='bin_numbers_bmi', 
                                                  data=data_combined, top=50)

In [ ]:
pred_bmisds, pred_score_age, coef_bmisds = get_prediction_score('z_BMI.Nysom', data_combined, proteins, features_bmisds, 'bin_numbers_bmi')

In [ ]:
sns.scatterplot(x='z_BMI.Nysom', y='y_pred', data=pred_bmisds)

In [ ]:
# Get prediction values and scores for age
pred_age, pred_score_age, coef_age = get_prediction_score('age', data_combined, proteins, features_age, 'age_int')

# Get prediction values and scores for BMI
pred_bmi, pred_score_bmi, coef_bmi = get_prediction_score('BMI', data_combined, proteins, features_bmi, 'bin_numbers_bmi')

In [ ]:
# Get prediction values and scores for age using top 5 features
pred_age_top5, pred_score_age_top5, coef_age_top5 = get_prediction_score('age', data_combined, proteins, features_age[:5], 'age_int')

In [ ]:
x='age'
y='y_pred'
data=pred_age_top5
score = pred_score_age_top5
fig, ax=plt.subplots()
h=sns.scatterplot(ax=ax, x=x, y=y, data=data, color='royalblue', edgecolor='white')
h=sns.regplot(ax=ax, x=x, y=y, data=data, scatter=False, color='darkred')
h.set(xlim=(4.6, 20.6), ylim=(4.6,20.6), xticks=np.arange(3,11)*2)
h.annotate('Test dataset (n={})\nPearson: {}\nMean absolute error:{}'.format(data.shape[0],score.loc['Pearson r'][x], score.loc['mean_absolute_error'][x]), 
           xycoords='axes fraction', xy=(0.02, 0.8), fontsize=13)

### Export supplementary tables

In [ ]:
df_table4_age = pd.DataFrame.from_dict({'predictors for age':features_age, 
                                       'coefs':coef_age})
df_table4_bmi = pd.DataFrame.from_dict({'predictors for BMI':features_bmi, 
                                       'coefs':coef_bmi})
df_table4 = pd.concat([df_table2_age, df_table2_bmi], ignore_index=True, axis=1)
df_table4.columns = ['predictors for age (UniprotID_Genename)', 'coefficient', 
                     'predictors for BMI (UniprotID_Genename)','coefficient' ]

In [ ]:
df_table3 = pd.read_csv(folders['tables'] / 'protein_age_clusters.txt', sep='\t')

In [ ]:
cluster_dict = {'Cluster -156':'Cluster1', 'Cluster -152':'Cluster2', 
                'Cluster -142':'Cluster3','Cluster -149':'Cluster4', 
                'Cluster -124':'Cluster5', 'Cluster -99':'Cluster6', 
                'Cluster -162':'Cluster7'}

In [ ]:
df_table3['Cluster2'] = df_table3['Cluster'].map(cluster_dict)
df_table3['Cluster2']=df_table3['Cluster2'].fillna('Unclassified')
df_table3 = df_table3[1:].drop(['Cluster'], axis=1).sort_values(by='Cluster2').rename({'Cluster2':'Cluster'}, axis=1)
df_table3 = df_table3.reset_index().drop('index', axis=1)
df_table3 = df_table3.set_index(['Cluster', 'Protein ID', 'Gene name']).astype(np.float)

In [ ]:
new_index = df_table3.index
new_cols = pd.MultiIndex.from_tuples([('boys', i) for i in np.arange(5, 20)] + 
                                     [('girls', i) for i in np.arange(5, 21)])

In [ ]:
df_table3_formatted = pd.DataFrame(df_table3.values, index=new_index, columns=new_cols)
df_table3_formatted = df_table3_formatted.rename_axis(['sex', 'age'], axis=1)

In [ ]:
df_tablestudy = pd.read_excel(folders['tables'] / 'TableS6.xlsx', engine='openpyxl')

In [ ]:
# # Write table 1-3, and 6 to supplementary tables
# with pd.ExcelWriter('tables/tableS2_3_4_7.xlsx') as writer:
#     df_table2_formatted.to_excel(writer,sheet_name='ST2', index=False)
#     df_table3_formatted.to_excel(writer,sheet_name='ST3', index=False)
#     df_table4.to_excel(writer,sheet_name='ST4', index=False)
#     df_tablestudy.to_excel(writer, sheet_name='ST7', index=False)
#     #df_table3_formatted.to_excel(writer, sheet_name='ST5')

### Figure 2

In [ ]:
from venn import venn

In [ ]:
groups = sig_stats.groupby('names')
res = {key1:group['dep_var'].tolist() for key1, group in groups}
res = {key:set(res[key]) for key in ['age', 'sex', 'z_BMI.Nysom', 'overweight*z_BMI.Nysom']}

In [ ]:
sns.set_style('ticks')
fig, axes = plt.subplots(3,3, figsize=(16,16))
FACTORS = ['age', 'z_BMI.Nysom', 'sex']
FACTORS_4= ['age', 'z_BMI.Nysom', 'sex', 'overweight*z_BMI.Nysom']
dfs=[lireg_dicts[i]['df'] for i in FACTORS]
axes[1, 0].set_title('Age associated proteome', fontsize=14)
axes[1, 1].set_title('BMI SDS and overweight*BMI SDS associated proteome', fontsize=14)
axes[1, 2].set_title('Sex associated proteome', fontsize=14)
axs = axes.flat
for n, ax in enumerate(axs):
    ax.text(-0.2, 1.1, string.ascii_lowercase[n], transform=ax.transAxes, size=15, weight='bold')

x='coef'
y='-Log10 P-value'
palette={'not significant':'gray', 'pos':'darkred', 'neg':'royalblue'}
palette2={'pos_z_BMI.Nysom':'pink', 'pos_overweight*z_BMI.Nysom':'darkred', 
          'neg_z_BMI.Nysom':'royalblue', 'neg_overweight*z_BMI.Nysom':'darkblue', 
          'not significant':'gray'}

# Age associated
df=dfs[0].reset_index()
texts=[]
a=sns.scatterplot(ax=axes[1, 0], x=x, y=y, data=df, hue='direction', 
                size=y, palette=palette, legend=False, alpha=1, edgecolor='white')
axes[1,0].set_xlim(-1, 1)
for i in range(len(df)):
    if (df.iloc[i][y] > 50) and (abs(df.iloc[i][x])>0.1):
        texts.append(axes[1, 0].text(x=df.iloc[i][x], y=df.iloc[i][y], s=df.iloc[i]['Gene name']))

for index, row in igfs.iterrows():
    texts.append(axes[1, 0].text(x=row[x], y=row[y], s=row['Gene name']))

adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.5), ax=axes[1, 0])

# BMI associated
df=dfs[1].reset_index()
texts=[]
a=sns.scatterplot(ax=axes[1, 1], x=x, y=y, data=df, hue='direction', 
                size=y, palette=palette, legend=False, alpha=1, edgecolor='white', )
axes[1,1].set_xlim(-1, 1)
for i in range(len(df)):
    if (df.iloc[i][y] > 5) and (abs(df.iloc[i][x])>0.2):
        texts.append(axes[1, 1].text(x=df.iloc[i][x], y=df.iloc[i][y], s=df.iloc[i]['Gene name']))
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.5), ax=axes[1, 1])

sns.scatterplot(ax=axes[1, 1], x=x, y=y, data=df, hue='direction2',
                size=y,palette=palette2, legend=False, alpha=1, edgecolor='white')
axes[1,1].set_xlim(-1, 1)

# Sex associated 
df=dfs[2].reset_index()
texts=[]
a=sns.scatterplot(ax=axes[1, 2], x=x, y=y, data=df, hue='direction', 
                size=y, palette=palette, legend=False, alpha=1, edgecolor='white')
axes[1,2].set_xlim(-1, 1)
for i in range(len(df)):
    if (df.iloc[i][y] > 20) and (abs(df.iloc[i][x])>0.3):
        texts.append(axes[1, 2].text(x=df.iloc[i][x], y=df.iloc[i][y], s=df.iloc[i]['Gene name']))
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.5), ax=axes[1, 2])

for i in range(3):
    for j in range(3):        
        axes[i, j].tick_params(axis='x', labelsize=13)
        axes[i, j].tick_params(axis='y', labelsize=13)
        axes[i, j].xaxis.label.set_size(fontsize=13)
        axes[i, j].yaxis.label.set_size(fontsize=13)

venn(res, ax=axes[0,1], cmap='vlag', alpha=0.6, )

c=sns.stripplot(ax=axes[0,2], x='names', y='coef', data=sig_stats[sig_stats['names'].isin(FACTORS_4)], 
                palette = dict(zip(FACTORS_4, ['white', 'royalblue', 'gray','pink'])), 
                       alpha=1, edgecolor='black', linewidth=0.5, size=5)

x='age'
y='y_pred'
data=pred_age
score = pred_score_age
h=sns.scatterplot(ax=axes[2,1], x=x, y=y, data=data, color='royalblue', edgecolor='white')
h=sns.regplot(ax=axes[2,1], x=x, y=y, data=data, scatter=False, color='darkred')
h.set(xlim=(4.6, 20.6), ylim=(4.6,20.6), xticks=np.arange(3,11)*2)
h.annotate('Test dataset (n={})\nPearson: {}\nMean absolute error:{}'.format(data.shape[0],score.loc['Pearson r'][x], score.loc['mean_absolute_error'][x]), 
           xycoords='axes fraction', xy=(0.02, 0.8), fontsize=13)
x='BMI'
#x='z_BMI.Nysom'
data=pred_bmi
score = pred_score_bmi
i=sns.scatterplot(ax=axes[2,2], x=x, y=y, data=data, color='royalblue', edgecolor='white')
i=sns.regplot(ax=axes[2,2], x=x, y=y, data=data, scatter=False, color='darkred')
i.set(xlim=(10, 50), ylim=(10,50))
i.annotate('Test dataset (n={})\nPearson: {}\nMean absolute error:{}'.format(data.shape[0],score.loc['Pearson r'][x], score.loc['mean_absolute_error'][x]), 
           xycoords='axes fraction', xy=(0.02, 0.8), fontsize=13)



axes[0, 0].axis('off')
axes[2, 0].axis('off')
plt.rcParams['pdf.fonttype'] = 42
plt.subplots_adjust(wspace=0.3, hspace=0.3)
plt.savefig('figures/figure 2.pdf', dpi=120, bbox_inches='tight')

### Figure 2 - Trajectories of age associated proteins
- export to Perseus to generate heatmap

In [ ]:
sig_age = lireg_dicts['age']['sig_set']
#sig_puberty = lireg_dicts['pubertal_status2']['sig_set']
sig_age_proteins = list(set(proteins) & set(sig_age))
df_age = data_combined.groupby(['sex', 'age_int'])[proteins].median()[sig_age_proteins].T.dropna(axis=0)
df_age['Gene name']=df_age.index.str.split('_').str[1]
df_age['Protein ID']=df_age.index.str.split('_').str[0]

df_sig_age = lireg_dicts['age']['sig']
df_sig_age.loc[:, 'abs(coef)']=abs(df_sig_age['coef'])
toplot_proteins = df_sig_age[df_sig_age['abs(coef)']>0.06]['dep_var'].tolist()

#df_age.to_csv('data/processed/age_sig.csv')
#df_age.loc[toplot_proteins].to_csv('data/processed/age_sig_coef.csv')

#### Plot examples

In [ ]:
toplot=['P04278_SHBG', 'P01023_A2M', 'P06276_BCHE', 'P35443_THBS4',
       'P02741_CRP', 'P18428_LBP', 'P02790_HPX', 'P04004_VTN', 'P35858_IGFALS','P17936_IGFBP3', 'P00747_PLG',
        'P20742_PZP', 'P01019_AGT', 'P05019_IGF1']

In [ ]:
df_toplot = data_combined.copy().dropna(subset=['age', 'sex'])
df_toplot['sex_obesity'] = df_toplot['sex'].astype(str) + '_' + df_toplot['obesity'].astype(str)

In [ ]:
cli_to_include = ['ALAT', 'ASAT', 'GGT', 'glucose', 'insulin', 'HbA1c', 'triglycerides', 
                 'chol_total', 'chol_ldl', 'chol_hdl', 'bp_sys', 'bp_dia', 
                 'IgA', 'IgG', 'IgM','hs_CRP_SSI']
cli_to_include = ['insulin']

In [ ]:
aaa = df_toplot.groupby(['sex', 'age_int', ])[toplot_proteins+cli_to_include].median().T

In [ ]:
import dash_bio

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist
data=aaa
standardized_data = data.sub(data.mean(axis=1), axis=0).div(data.std(axis=1), axis=0)

Z = linkage(standardized_data, method='average')
row_order = leaves_list(Z)
reordered_data = data.iloc[row_order][::-1]

In [ ]:
down_list = reordered_data[71:].index.str.split('_').str[1].dropna().tolist()
up_list = reordered_data[:42].index.str.split('_').str[1].dropna().tolist()

In [ ]:
data=aaa
custom_color_map = ['#0000ff', '#ffffff', '#ff0000']
import plotly.graph_objects as go
import plotly.io as pio
dash_bio.Clustergram(
    data=data,
    column_labels=list(data.columns.values),
    row_labels=list(data.index),
    height=800,
    standardize='row',
    cluster='row',
    width=700, 
    color_list=colors[:2],
    color_map = custom_color_map, 
    link_method='average', 
)

#### Source data for Extended Data Fig. 1

In [ ]:
sourcedata_edfig1 = aaa.apply(lambda x: (x-x.mean())/x.std(), axis=1).iloc[g.dendrogram_row.reordered_ind]
sourcedata_edfig1.to_excel('source_data/SourceData_ExtendedDataFigure1.xlsx', sheet_name='Extended Data Fig. 1a')

In [ ]:
g=sns.clustermap(aaa, z_score=0, cmap='bwr', col_cluster=False, vmin=-4, vmax=4, yticklabels=True)
plt.rcParams['pdf.fonttype'] = 42
plt.savefig('figures/heatmap.pdf')%%sh

In [ ]:
background_genes = [i.split('_')[1] for i in proteins]

In [ ]:
toplot = ['P05019_IGF1', 'P24593_IGFBP5', 'P08833_IGFBP1', 'P18065_IGFBP2',
           'Q15848_ADIPOQ', 'P04278_SHBG', 'P01023_A2M','P02741_CRP', 
         'P16112_ACAN', 'P02452_COL1A1',  'P35443_THBS4','Q15063_POSTN',
          'Q9Y5C1_ANGPTL3','P05556_ITGB1', 'P05362_ICAM1', 'P12821_ACE',
         
         
          'P27487_DPP4', 'P98160_HSPG2', 'P02743_APCS', 
         'P01019_AGT', 'P20742_PZP',]

In [ ]:
df_toplot.dropna(subset=['sex', 'obesity']).groupby(['sex', 'obesity']).count()

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(10,8))
fig.subplots_adjust(hspace=0.6, wspace=0.8)
n=1
for (ax, protein) in zip(axs.flat, toplot):
    ax.text(-0.7, 1.05, string.ascii_lowercase[n], transform=ax.transAxes, weight='bold')
    n+=1
    sns.lineplot(x='age_int', y=protein, data=df_toplot, 
                 hue='sex', style='obesity', palette=['#00539CFF', '#ED2B33FF',], ax=ax, legend=False)
    genename=protein.split('_')[1]
    ax.set_title(genename)
    ax.set_xlabel('')
    ax.set_ylabel('MS signal\n[Log2]')
    ax.set_xticks([5*i for i in np.arange(1, 5)])
    ax.set_ylim(ax.get_ylim()[0]*1, ax.get_ylim()[1]*1)
    
plt.rcParams['pdf.fonttype'] = 42
fig.savefig('figures/figure2.pdf', dpi=120, bbox_inches='tight')

In [ ]:
import gseapy as gp
from gseapy import barplot, dotplot
gp.get_library_name()
pathways = ['GO_Biological_Process_2023']

In [ ]:
age_enrichr = gp.enrichr(gene_list=age_genes, # or "./tests/data/gene_list.txt",
                 gene_sets=pathways,
                 organism='human', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir=None, # don't write to disk
                )

In [ ]:
list_tocheck = [i.split('_')[1] for i in toplot_proteins]

In [ ]:
enr_down = gp.enrichr(gene_list=down_list, # or "./tests/data/gene_list.txt",
                 gene_sets=pathways,
                 organism='human', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir=None, # don't write to disk
                )

In [ ]:
enr_up = gp.enrichr(gene_list=up_list, # or "./tests/data/gene_list.txt",
                 gene_sets=pathways,
                 organism='human', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir=None, # don't write to disk
                )

protein = 'height'

In [ ]:
protein='P05186_ALPL'

In [ ]:
fig, ax=plt.subplots(figsize=(4,4))
sns.lineplot(x='age_int', y=protein, data=df_toplot, 
             hue='sex',  style='obesity',palette=['green', 'gray'], legend=True)